In [1]:
## %% [markdown]
# # 04. Building the final dataset
# Final stage: merging literature and experimental records, filling empty cells,
# consistency check, element statistics.



In [2]:
from pathlib import Path
import numpy as np
import pandas as pd

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = BASE_DIR / "data" / "raw"
TABLES_DIR = BASE_DIR / "outputs" / "tables"

LIT_PATH = RAW_DIR / "database_row.xlsx"
EXP_PATH = RAW_DIR / "experimental_results.xlsx"
FINAL_PATH = BASE_DIR / "database_final.xlsx"
STATS_PATH = TABLES_DIR / "final_dataset_element_statistics.xlsx"

TABLES_DIR.mkdir(parents=True, exist_ok=True)

ELEMENTS = ['Ni', 'Al', 'Co', 'Cr', 'Cu', 'Fe', 'Ti',
            'V', 'Mo', 'Hf', 'W', 'Ta', 'B', 'C', 'Nb', 'Mn']
ZERO_COLS = ["HT 1, °C", "HT 1, min", "Recrystallization, °C", "Recrystallization, min",
             "Aging 1, °C", "Aging 1, min", "Aging 2, °C", "Aging 2, min",
             "Cold deformation, %"]
NAN_COLS_NUM = ["Solvus Temperature, °C", "L12 fraction, %"]
NAN_COLS_TXT = ["Cooling_after_HT1", "Cooling_after_Recryst", "Cooling_after_Aging1"]
NAN_COLS = NAN_COLS_NUM + NAN_COLS_TXT

# Raise an error on L12 inconsistencies (True) or only report them (False)
STRICT_L12_CHECK = False


In [3]:
# 1. Loading both files
df_lit = pd.read_excel(LIT_PATH)
df_exp = pd.read_excel(EXP_PATH)

for df_ in (df_lit, df_exp):
    df_.columns = [c.strip() for c in df_.columns]

print(f"Literature records: {len(df_lit)}, columns: {df_lit.shape[1]}")
print(f"Experimental records: {len(df_exp)}, columns: {df_exp.shape[1]}")


Literature records: 374, columns: 38
Experimental records: 12, columns: 38


In [4]:
# 2. Column consistency check (column order may differ, names must match)
only_lit = [c for c in df_lit.columns if c not in df_exp.columns]
only_exp = [c for c in df_exp.columns if c not in df_lit.columns]
assert not only_lit and not only_exp, (
    f"Column names differ.\nOnly in database_row: {only_lit}\n"
    f"Only in experimental_results: {only_exp}"
)

# Experimental columns follow the literature column order
df_exp = df_exp[df_lit.columns]


In [5]:
# 3. Appending the experimental records below the literature table
df = pd.concat([df_lit, df_exp], ignore_index=True)
assert len(df) == len(df_lit) + len(df_exp)
assert list(df.columns) == list(df_lit.columns)

print(f"Merged dataset: {len(df)} records "
      f"({len(df_lit)} literature + {len(df_exp)} experimental), {df.shape[1]} columns")

needed = (ELEMENTS + ZERO_COLS + NAN_COLS +
          ['Paper', 'DOI', 'Phase composition', 'Target state', 'L12 presence'])
missing = [c for c in needed if c not in df.columns]
assert not missing, f"Missing columns: {missing}\nAvailable: {list(df.columns)}"


Merged dataset: 386 records (374 literature + 12 experimental), 38 columns


In [6]:
# 4. Filling empty cells
# Cells containing only whitespace are treated as empty
fill_cols = ZERO_COLS + NAN_COLS
df[fill_cols] = df[fill_cols].replace(r"^\s*$", np.nan, regex=True)

# Numeric validation before filling
el_num = df[ELEMENTS].apply(pd.to_numeric, errors='coerce')
assert not (df[ELEMENTS].notna() & el_num.isna()).any().any(), "Non-numeric values in elements"
assert not el_num.isna().any().any(), \
    f"Empty element cells: {el_num.isna().sum()[lambda s: s > 0].to_dict()}"

pr_num = df[ZERO_COLS].apply(pd.to_numeric, errors='coerce')
assert not (df[ZERO_COLS].notna() & pr_num.isna()).any().any(), "Non-numeric values in heat-treatment parameters"

opt_num = df[NAN_COLS_NUM].apply(pd.to_numeric, errors='coerce')
assert not (df[NAN_COLS_NUM].notna() & opt_num.isna()).any().any(), "Non-numeric values in optional numeric fields"

# Processing steps that were not applied -> 0
df[ELEMENTS] = el_num
df[ZERO_COLS] = pr_num.fillna(0)
assert not df[ELEMENTS + ZERO_COLS].isna().any().any()

# L12 consistency check uses the numeric values BEFORE the visible "NaN" text is written
l12_frac = opt_num["L12 fraction, %"]


In [7]:
# 5. Consistency check: L12 fraction vs L12 presence
l12_presence = pd.to_numeric(df["L12 presence"], errors="coerce")
assert not (df["L12 presence"].notna() & l12_presence.isna()).any(), \
    "Non-numeric values in 'L12 presence'"

# Rows with an unreported fraction (NaN) are excluded: comparisons with NaN are False
bad_zero = df[(l12_frac == 0) & (l12_presence == 1)]  # fraction measured as 0, but L12 is marked as present
bad_pos = df[(l12_frac > 0) & (l12_presence == 0)]    # fraction > 0, but L12 is marked as absent

print(f"L12 fraction = 0 with L12 presence = 1: {len(bad_zero)}")
print(f"L12 fraction > 0 with L12 presence = 0: {len(bad_pos)}")
if len(bad_zero):
    print("\nPapers with fraction = 0 and L12 present:")
    print(bad_zero[["Paper", "DOI"]].drop_duplicates().to_string(index=False))
if len(bad_pos):
    print("\nPapers with fraction > 0 and L12 absent:")
    print(bad_pos[["Paper", "DOI"]].drop_duplicates().to_string(index=False))

if STRICT_L12_CHECK:
    assert len(bad_zero) == 0 and len(bad_pos) == 0, "L12 fraction / presence inconsistencies found"


L12 fraction = 0 with L12 presence = 1: 0
L12 fraction > 0 with L12 presence = 0: 0


In [8]:
# 6. Saving the final dataset (visible "NaN" text for unreported values)
df_final = df.copy()
df_final[NAN_COLS] = df_final[NAN_COLS].astype(object).fillna("NaN")
df_final.to_excel(FINAL_PATH, index=False)
print(f"Saved: {FINAL_PATH} | {df_final.shape[0]} rows, {df_final.shape[1]} columns")


Saved: C:\Users\Николай\PycharmProjects\FeNiCoAlCrTi-dataset\database_final.xlsx | 386 rows, 38 columns


In [9]:
# 7. Element statistics of the final dataset
el_stats = pd.DataFrame({
    'Records with element (>0)': (df[ELEMENTS] > 0).sum(),
    'Zero (element absent)': (df[ELEMENTS] == 0).sum(),
    'Min (>0), at.%': df[ELEMENTS].where(df[ELEMENTS] > 0).min(),
    'Max, at.%': df[ELEMENTS].max()}).round(2)
el_stats['Share of records, %'] = (
    el_stats['Records with element (>0)'] / len(df) * 100
).round(1)

assert (el_stats['Records with element (>0)'] + el_stats['Zero (element absent)'] == len(df)).all()

el_stats = el_stats.loc[ELEMENTS].rename_axis('Element').reset_index()
print(el_stats.to_string(index=False))

with pd.ExcelWriter(STATS_PATH, engine='openpyxl') as writer:
    el_stats.to_excel(writer, sheet_name='Elements', index=False)
    ws = writer.sheets['Elements']
    for col_cells in ws.columns:
        width = max(len(str(c.value or '')) for c in col_cells)
        ws.column_dimensions[col_cells[0].column_letter].width = min(max(width + 3, 10), 60)
print(f"Statistics saved: {STATS_PATH}")

Element  Records with element (>0)  Zero (element absent)  Min (>0), at.%  Max, at.%  Share of records, %
     Ni                        382                      4            11.2       60.8                 99.0
     Al                        362                     24             1.0       25.0                 93.8
     Co                        359                     27             4.5       61.0                 93.0
     Cr                        365                     21             0.6       32.0                 94.6
     Cu                         26                    360             1.0       18.2                  6.7
     Fe                        372                     14             1.3       55.0                 96.4
     Ti                        292                     94             1.0       20.0                 75.6
      V                         52                    334             3.0        6.0                 13.5
     Mo                         25            